# Paimon INTERNAL via Flink example notebook

Registers and validates PAIMON INTERNAL metadata through Kasanari REST APIs.

In [ ]:
import json
import os
import urllib.request

base_url = os.environ.get("KASANARI_BASE_URL", "http://kasanari:9090")
jdbc_uri = os.environ.get("KASANARI_JDBC_URI", "jdbc:postgresql://catalog-storage:5432/postgres")
s3_endpoint = os.environ.get("KASANARI_S3_ENDPOINT", "http://minio:9000")
catalog_id = "paimon_flink_internal"

payload = {
    "catalogId": catalog_id,
    "catalogType": "PAIMON",
    "mode": "INTERNAL",
    "spec": {
        "fileIoProperties": {
            "fs.s3a.access.key": "admin",
            "fs.s3a.secret.key": "password",
            "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "fs.s3a.path.style.access": "true",
            "fs.s3a.endpoint": s3_endpoint,
        },
        "catalogProperties": {
            "warehouse": "s3a://warehouse",
            "uri": jdbc_uri,
            "kasanari.jdbc.user": "postgres",
            "kasanari.jdbc.password": "postgres",
            "kasanari.catalog.key": catalog_id,
        },
    },
}

request = urllib.request.Request(
    f"{base_url}/management/v1/catalogs",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request) as response:
    print("register status:", response.status)


In [ ]:
with urllib.request.urlopen(f"{base_url}/management/v1/catalogs/PAIMON/{catalog_id}") as response:
    body = json.loads(response.read().decode("utf-8"))
    print("fetch status:", response.status)
    print("catalog:", body.get("catalogId"), "mode:", body.get("mode"))
